# AeroPulse Source Likelihood — 01: Data Contract, Quality & Provenance

Import the evidence table, prove it meets a contract, and record where every
piece of evidence came from and how much to trust it.

## Objective

Improvement plan sections 9-12 and 38.

The previous Phase 1 imported the pm25 table, checked eight columns existed
and wrote a fingerprint. That is a reproducibility foundation, not a data
contract. This notebook adds what the plan asks for:

- a machine-readable contract with units and physical ranges
- quality gates (duplicates, ranges, coverage)
- **evidence provenance** — what each family actually is, its latency, and a
  reliability weight, because CAMS model output and a CPCB measurement are
  not the same kind of fact
- **availability flags per evidence family**, so a labeling function can tell
  "no fire was detected" from "the fire feed was unavailable"

One finding shapes everything downstream and is measured in section 3: the
co-pollutants the old notebook fetched optionally from OpenAQ (default off)
are **already present** in the upstream table as CAMS fields, at full
coverage. That removes the network dependency entirely — and introduces a
circularity that section 4 states plainly.


> **Scientific constraint, repeated in every notebook because it governs how
> the output may be used.** These are *likelihoods derived from heuristics*,
> not measured source attribution. No public ground-truth attribution
> catalogue exists for this dataset. A model that scores well here has
> reproduced its own labeling system — which is not the same as being right.

In [1]:
# ============================================================================
# SETUP — reusable logic lives in source_toolkit.py, not in cells.
# ============================================================================
import os, sys, json, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd()))
import source_toolkit as st

warnings.filterwarnings("ignore")
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 60)

cfg = st.load_config()
SOURCES = st.SOURCES
print("kernel:", sys.executable)

env: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/source_likelihood/.env
project: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/source_likelihood
sources: ['biomass_burning', 'traffic', 'industrial', 'dust', 'regional_transport']
kernel: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/.venv/bin/python


## 1. Load the upstream evidence table

In [2]:
t0 = time.time()
if not cfg.event_aware_file.exists():
    raise FileNotFoundError(
        f"{cfg.event_aware_file} not found — run the pm25_estimator pipeline first."
    )
df = pd.read_parquet(cfg.event_aware_file)
df["timestamp_utc"] = pd.to_datetime(df["timestamp_utc"], utc=True)

print(f"loaded in {time.time() - t0:.1f}s")
print("rows     ", f"{len(df):,}")
print("columns  ", df.shape[1])
print("stations ", df["location_id"].nunique())
print("window   ", df["timestamp_utc"].min(), "->", df["timestamp_utc"].max())

loaded in 0.3s
rows      1,705,252
columns   208
stations  149
window    2025-02-18 20:30:00+00:00 -> 2026-09-09 07:30:00+00:00


## 2. Contract and quality gates

In [3]:
contract = st.validate_contract(df)
print(json.dumps(contract, indent=2))

broken = df.drop(columns=["wind_direction_10m"]).head(500)
try:
    st.validate_contract(broken)
    print("FAIL — validator accepted a dataset with no wind direction")
except ValueError as exc:
    print("\nPASS — contract rejected the broken dataset:", exc)

{
  "schema_version": "2.0.0",
  "label_schema_version": "2.0.0",
  "missing_required": [],
  "absent_evidence": [],
  "rows": 1705252,
  "columns": 208,
  "status": "PASS"
}

PASS — contract rejected the broken dataset: Data contract violated. missing_required=['wind_direction_10m']


In [4]:
report, coverage = st.quality_report(df)
core = {k: v for k, v in report.items() if k not in ("evidence_missing_pct", "out_of_range")}
print(json.dumps(core, indent=2))
print("\nout of range:", report["out_of_range"] or "none")
print("\nevidence missingness (%):")
for name, pct in sorted(report["evidence_missing_pct"].items(), key=lambda kv: -kv[1]):
    print(f"  {name:<26} {pct:6.2f}")
print("\nQUALITY STATUS:", report["quality_status"])

{
  "generated_utc": "2026-09-10T15:51:53.560161+00:00",
  "rows": 1705252,
  "stations": 149,
  "window_start": "2025-02-18 20:30:00+00:00",
  "window_end": "2026-09-09 07:30:00+00:00",
  "duplicate_rows": 0,
  "missing_pm25_pct": 0.0,
  "failures": [],
  "quality_status": "PASS"
}

out of range: none

evidence missingness (%):
  upwind_pm25_mean            36.29
  cams_pm10                    0.00
  cams_no2                     0.00
  cams_co                      0.00
  cams_so2                     0.00
  cams_o3                      0.00
  cams_dust                    0.00
  upwind_fire_frp_50km         0.00
  boundary_layer_height        0.00

QUALITY STATUS: PASS


## 3. Co-pollutants: already here, and not what the plan assumed

The old Phase 1 treated NO2/CO/SO2/O3/PM10 as an optional OpenAQ fetch,
default off, and warned that "traffic labels will be weaker" without it.
Check what the upstream table actually carries.

In [5]:
copollutants = ["cams_no2", "cams_co", "cams_so2", "cams_o3", "cams_pm10", "cams_dust"]
present = [c for c in copollutants if c in df.columns]
summary = df[present].describe().T[["mean", "50%", "max"]].round(2)
summary["null_pct"] = (df[present].isna().mean() * 100).round(3)
print(summary.to_string())

legacy = [c for c in ("no2", "co", "so2", "o3", "pm10") if c in df.columns]
print("\nmeasured (OpenAQ) co-pollutant columns present:", legacy or "none")
print("SOURCE_ENABLE_COPOLLUTANTS:", cfg.enable_copollutants,
      "-> the network fetch is unnecessary; CAMS already covers every row")

             mean    50%     max  null_pct
cams_no2    13.11    9.2   220.1     0.000
cams_co    359.28  296.0  4984.0     0.000
cams_so2    13.35    9.4   162.9     0.000
cams_o3     87.74   78.0   523.0     0.000
cams_pm10   73.81   42.6  3270.8     0.003
cams_dust   68.38   12.0  9119.0     0.000

measured (OpenAQ) co-pollutant columns present: none
SOURCE_ENABLE_COPOLLUTANTS: False -> the network fetch is unnecessary; CAMS already covers every row


## 4. Provenance — and the circularity this creates

CAMS is not a measurement. It is a chemical-transport model whose output is
driven by an **emissions inventory**, which already encodes assumptions about
where traffic and industry are. Using `cams_no2` as evidence for a traffic
source therefore partly reads back CAMS's own traffic inventory.

That does not make it useless — the inventory encodes real information — but
it does mean a traffic likelihood built on CAMS is weaker than its number
suggests, and it cannot be treated as independent confirmation. The
reliability weights below carry that discount into the labeling functions,
where it belongs.

In [6]:
for family, meta in st.EVIDENCE_PROVENANCE.items():
    print(f"{family:<22} {meta['kind']:<18} latency {meta['latency_minutes']:>4} min   "
          f"reliability {meta['reliability']:.2f}   [{meta['measurement_source']}]")
    if "caveat" in meta:
        print(f"    caveat: {meta['caveat']}")

pm25                   measurement        latency   60 min   reliability 0.95   [CPCB/OpenAQ]
cams_*                 model_analysis     latency  300 min   reliability 0.55   [CAMS (Open-Meteo air quality)]
    caveat: CAMS is chemical-transport model output driven by an emissions inventory, not a measurement. Using cams_no2 as traffic evidence partly reads back CAMS's own traffic inventory -- treat any co-pollutant-driven likelihood as weaker than its number suggests.
fire_*                 satellite_detection latency  180 min   reliability 0.90   [NASA FIRMS (VIIRS/MODIS)]
weather_*              reanalysis         latency   60 min   reliability 0.85   [Open-Meteo / ERA5]
upwind_*               derived            latency   60 min   reliability 0.75   [derived from the station network]


## 5. Evidence availability flags

Plan section 38: "no fire evidence" and "fire data unavailable" are
completely different claims. Without a flag distinguishing them, a labeling
function reading a NaN has no way to abstain rather than vote against.

In [7]:
df = st.add_evidence_availability(df)
flags = [c for c in df.columns if c.startswith("evidence_") and c.endswith("_available")]
print("availability by family:")
for flag in flags:
    print(f"  {flag:<38} {float(df[flag].mean()):6.2%} of rows")

print(f"\nevidence_coverage: mean {df['evidence_coverage'].mean():.3f}, "
      f"p10 {df['evidence_coverage'].quantile(0.10):.3f}")
print("\nevidence quality bands:")
print(st.evidence_quality(df["evidence_coverage"]).value_counts().sort_index().to_string())

availability by family:
  evidence_pm25_available                100.00% of rows
  evidence_copollutant_available         100.00% of rows
  evidence_fire_available                100.00% of rows
  evidence_weather_available             100.00% of rows
  evidence_upwind_available              63.71% of rows
  evidence_satellite_available           100.00% of rows

evidence_coverage: mean 0.927, p10 0.800

evidence quality bands:
evidence_coverage
LOW             0
MEDIUM          2
HIGH      1705250


## 6. Persist

In [8]:
float_cols = df.select_dtypes(include=["float64"]).columns
df[float_cols] = df[float_cols].astype("float32")
out = cfg.base_dir / "source_evidence.parquet"
df.to_parquet(out, index=False)
coverage.to_csv(cfg.base_dir / "station_coverage.csv", index=False)

fingerprint = st.dataset_fingerprint(df[["timestamp_utc", "location_id", "pm25"]])
report["fingerprint"] = fingerprint
(cfg.base_dir / "data_quality_report.json").write_text(json.dumps(report, indent=2, default=str))

manifest = {
    "schema_version": cfg.schema_version,
    "label_schema_version": st.SCHEMA["label_schema_version"],
    "generated_utc": pd.Timestamp.now(tz="UTC").isoformat(),
    "source_path": str(cfg.event_aware_file),
    "rows": int(len(df)),
    "stations": int(df["location_id"].nunique()),
    "window": [str(df["timestamp_utc"].min()), str(df["timestamp_utc"].max())],
    "contract": contract,
    "quality_status": report["quality_status"],
    "provenance": st.EVIDENCE_PROVENANCE,
    "copollutant_source": "CAMS via Open-Meteo (model analysis, not measurement)",
    "fingerprint": fingerprint,
    "git_commit": st.git_commit(),
    "caveat": "weak-supervision evidence only; not source attribution",
}
(cfg.base_dir / "dataset_manifest.json").write_text(json.dumps(manifest, indent=2, default=str))
print("01 COMPLETE ->", out)
print("Next: 02_source_evidence_features.ipynb")

01 COMPLETE -> /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/source_likelihood/data/source/processed/base/source_evidence.parquet
Next: 02_source_evidence_features.ipynb


## Findings

**Contract and quality: PASS.** 1,705,252 rows, 149 stations, 2025-02-18 to
2026-09-09. All nine required columns and all nine evidence columns present,
no duplicates, no out-of-range values, zero missing PM2.5. The negative test
confirms the validator actually rejects a table missing `wind_direction_10m`.

**The optional co-pollutant fetch is unnecessary and always was.** The old
Phase 1 treated NO2/CO/SO2/O3/PM10 as an OpenAQ call behind
`SOURCE_ENABLE_COPOLLUTANTS`, default off, and warned that "traffic labels
will be weaker" without it. In fact the upstream table already carries all
five as CAMS fields at **0.00% missingness**, plus `cams_dust` — a direct
dust indicator the plan's section 16 asks for as a "dust/satellite indicator
if available".

So the pipeline no longer has a network dependency, and the traffic and
industrial labeling functions have full-coverage evidence. That is the good
news.

**The bad news is what those fields are.** CAMS is a chemical-transport model
whose output is driven by an emissions inventory that already encodes where
traffic and industry are. Using `cams_no2` as evidence for a traffic source
therefore partly reads back CAMS's own traffic inventory. It is not
independent confirmation, and a traffic likelihood built on it is weaker
than its number looks.

This is recorded as a reliability weight (0.55 for `cams_*`, against 0.95
for a CPCB PM2.5 measurement and 0.90 for a FIRMS fire detection) which the
labeling functions carry through to the aggregated label. It is the honest
way to use the data without either discarding it or overtrusting it.

**Evidence availability is near-total except upwind.**

| family | available |
|---|---|
| PM2.5 | 100.00% |
| co-pollutant (CAMS) | 100.00% |
| fire | 100.00% |
| weather | 100.00% |
| **upwind neighbours** | **63.71%** |

Mean evidence coverage 0.927, p10 0.800. One row in three has no upwind
field, which is a station-density artefact rather than a feed outage — and
it is exactly the evidence the `regional_transport` hypothesis depends on.
The availability flag exists so those rows make their labeling functions
**abstain** rather than vote against transport. Plan section 38: "no fire
evidence" and "fire data unavailable" are completely different claims, and
without the flag a NaN silently becomes the second one.